In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from allensdk.brain_observatory.behavior.behavior_project_cache.\
    behavior_neuropixels_project_cache \
    import VisualBehaviorNeuropixelsProjectCache

%matplotlib inline

In [2]:
# Update this to a valid directory in your filesystem. This is where the data will be stored.
cache_dir = './data/'
cache = VisualBehaviorNeuropixelsProjectCache.from_s3_cache(cache_dir=cache_dir) #from_local_cache(cache_dir=cache_dir)

# get the metadata tables
units_table = cache.get_unit_table()
channels_table = cache.get_channel_table()
probes_table = cache.get_probe_table()
behavior_sessions_table = cache.get_behavior_session_table()
ecephys_sessions_table = cache.get_ecephys_session_table()

In [3]:
session_id = 1065437523 #1064644573
session = cache.get_ecephys_session(
            ecephys_session_id=session_id)

c:\Users\grace\anaconda3\envs\AMP3-Spring-2026-Project\Lib\site-packages\hdmf\spec\namespace.py:535: UserWarning: Ignoring cached namespace 'core' version 2.6.0-alpha because version 2.7.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."


In [7]:

session.stimulus_presentations.columns

Index(['stimulus_block', 'image_name', 'duration', 'start_time', 'end_time',
       'start_frame', 'end_frame', 'is_change', 'is_image_novel', 'omitted',
       'flashes_since_change', 'trials_id', 'stimulus_index', 'rewarded',
       'contrast', 'position_x', 'color', 'spatial_frequency',
       'is_sham_change', 'active', 'orientation', 'temporal_frequency',
       'stimulus_name', 'position_y'],
      dtype='object')

In [16]:
units = session.get_units()
channels = session.get_channels()
unit_channels = units.merge(channels, left_on='peak_channel_id', right_index=True)

#first let's sort our units by depth
unit_channels = unit_channels.sort_values('probe_vertical_position', ascending=False)

#now we'll filter them
good_unit_filter = ((unit_channels['snr']>1)&
                    (unit_channels['isi_violations']<1)&
                    (unit_channels['firing_rate']>0.1))
good_units = unit_channels.loc[good_unit_filter]


# Unit info
unit_indices = np.array(good_units.index)
spike_times = dict([(i,session.spike_times[i]) for i in unit_indices])

In [22]:
available_spike_ids = set(session.spike_times.keys())
unit_indicies = np.array([
    unit_id
    for unit_id in good_units.index
    if unit_id in available_spike_ids
    ])

In [23]:
spike_times = {
    i: session.spike_times[i]
    for i in unit_indicies
}

In [24]:
bin_size = 0.025

start_time = (
    session.stimulus_presentations.start_time.min()
)
end_time = (
    session.stimulus_presentations.end_time.max()
)

time_bins = np.arange(
    start_time,
    end_time + bin_size,
    bin_size
)

n_bins = len(time_bins)-1

Collecting all the features to get Y

In [25]:
neuron_id = unit_indicies[0]

In [26]:
y, _ = np.histogram(spike_times[neuron_id], bins = time_bins)

In [28]:
stim_table = session.stimulus_presentations.copy()
stim_table[['start_time', 'image_name', 'omitted']].head()

image_names = sorted(stim_table.image_name.dropna().unique())[:8]
X_visual = np.zeros((n_bins,9))

In [31]:
stim_table['omitted'] = (
    stim_table['omitted'].fillna(False)
)

In [32]:
for _, row in stim_table.iterrows():
    t=row.start_time
    bin_idx = np.searchsorted(time_bins, t)-1
    if bin_idx < 0 or bin_idx >=n_bins:
        continue
    if row.omitted:
        X_visual[bin_idx, -1] = 1

    else:
        img = row.image_name

        if img in image_names:
            img_idx = image_names.index(img)
            X_visual[bin_idx, img_idx] = 1

In [33]:
trials = session.trials

In [40]:
X_task = np.zeros((n_bins, 2))
for _, row in trials.iterrows():
    t=row.start_time
    bin_idx = np.searchsorted(time_bins, t) -1

    if bin_idx < 0 or bin_idx >= n_bins:
        continue
    if row.hit:
        X_task[bin_idx, 0] = 1
    
    if row.miss:
        X_task[bin_idx, 1] = 1

In [42]:
running = session.running_speed
running_interp = np.interp(time_bins[:-1], running.timestamps, running.speed)

lick_times = session.licks.timestamps.values
lick_counts, _ = np.histogram(lick_times, bins=time_bins)

eye = session.eye_tracking
pupil_interp = np.interp(time_bins[:-1], eye.timestamps, eye.pupil_area)

X_behavior = np.column_stack([lick_counts, running_interp, pupil_interp])

In [44]:
X_base = np.hstack([X_visual, X_task, X_behavior])
X_base.shape


(350225, 14)

In [45]:
def lagged_matrix(X, n_lags=10):
    lagged = []
    
    for lag in range(n_lags):
        shifted = np.roll(X, lag, axis=0)
        shifted[:lag] = 0
        lagged.append(shifted)

    return np.hstack(lagged)

In [46]:
X = lagged_matrix(X_base, n_lags=10)

In [ ]:
from sklearn.linear_model import